In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
!pip install ultralytics
!pip install segmentation_models_pytorch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.2 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
!cp /content/drive/MyDrive/images.zip /content/
!cp /content/drive/MyDrive/col_space_masks.zip /content/
!cp /content/drive/MyDrive/row_space_masks.zip /content/
!cp /content/drive/MyDrive/col_span_masks.zip /content/
!cp /content/drive/MyDrive/row_span_masks.zip /content/


In [ ]:
!unzip /content/images.zip -d /content/
!unzip /content/row_space_masks.zip -d /content/
!unzip /content/col_space_masks.zip -d /content/
!unzip /content/row_span_masks.zip -d /content/
!unzip /content/col_span_masks.zip -d /content/


Streaming output truncated to the last 5000 lines.
  inflating: /content/col_span_masks/train/table_84110.png  
  inflating: /content/col_span_masks/train/table_84111.png  
  inflating: /content/col_span_masks/train/table_84112.png  
  inflating: /content/col_span_masks/train/table_84113.png  
  inflating: /content/col_span_masks/train/table_84114.png  
  inflating: /content/col_span_masks/train/table_84115.png  
  inflating: /content/col_span_masks/train/table_84116.png  
  inflating: /content/col_span_masks/train/table_84122.png  
  inflating: /content/col_span_masks/train/table_84124.png  
  inflating: /content/col_span_masks/train/table_84125.png  
  inflating: /content/col_span_masks/train/table_84126.png  
  inflating: /content/col_span_masks/train/table_84127.png  
  inflating: /content/col_span_masks/train/table_84128.png  
  inflating: /content/col_span_masks/train/table_84129.png  
  inflating: /content/col_span_masks/train/table_84131.png  
  inflating: /content/col_span_mas

In [ ]:
def resize_pad(img, size=512, pad_val=255, is_mask=False):
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
    img_resized = cv2.resize(img, (new_w, new_h), interpolation=interp)

    if img.ndim == 3:
        canvas = np.full((size, size, 3), pad_val, dtype=img.dtype)
    else:
        canvas = np.full((size, size), pad_val, dtype=img.dtype)

    x0 = (size - new_w) // 2
    y0 = (size - new_h) // 2

    canvas[y0:y0+new_h, x0:x0+new_w] = img_resized
    return canvas, (x0, y0, new_w, new_h)



In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

IMG_SIZE = 512

class SpanTableDataset(Dataset):
    def __init__(
        self,
        img_dir,
        row_space_dir,
        col_space_dir,
        row_span_dir,
        col_span_dir,
        augment=True
    ):
        self.img_dir = img_dir
        self.row_space_dir = row_space_dir
        self.col_space_dir = col_space_dir
        self.row_span_dir = row_span_dir
        self.col_span_dir = col_span_dir
        self.augment = augment

        self.files = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        # =====================
        # 📷 LOAD IMAGE
        # =====================
        try:
          img = cv2.imread(os.path.join(self.img_dir, fname))
          img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

          # =====================
          # 🧱 LOAD SPACE MASKS
          # =====================
          row_space = cv2.imread(
              os.path.join(self.row_space_dir, fname),
              cv2.IMREAD_GRAYSCALE
          )
          col_space = cv2.imread(
              os.path.join(self.col_space_dir, fname),
              cv2.IMREAD_GRAYSCALE
          )

          # =====================
          # 🔗 LOAD SPAN MASKS
          # =====================
          row_span = cv2.imread(
              os.path.join(self.row_span_dir, fname),
              cv2.IMREAD_GRAYSCALE
          )
          col_span = cv2.imread(
              os.path.join(self.col_span_dir, fname),
              cv2.IMREAD_GRAYSCALE
          )

          # =====================
          # 🔀 SCALE AUGMENT
          # =====================
          if self.augment:
              scale = np.random.uniform(1.0, 1.2)
              h, w = img.shape[:2]

              img = cv2.resize(img, (int(w*scale), int(h*scale)), cv2.INTER_AREA)
              row_space = cv2.resize(row_space, (int(w*scale), int(h*scale)), cv2.INTER_NEAREST)
              col_space = cv2.resize(col_space, (int(w*scale), int(h*scale)), cv2.INTER_NEAREST)
              row_span  = cv2.resize(row_span,  (int(w*scale), int(h*scale)), cv2.INTER_NEAREST)
              col_span  = cv2.resize(col_span,  (int(w*scale), int(h*scale)), cv2.INTER_NEAREST)

          # =====================
          # 📐 RESIZE + PAD
          # =====================
          img, meta = resize_pad(img, IMG_SIZE, pad_val=255, is_mask=False)
          row_space, _ = resize_pad(row_space, IMG_SIZE, pad_val=0, is_mask=True)
          col_space, _ = resize_pad(col_space, IMG_SIZE, pad_val=0, is_mask=True)
          row_span,  _ = resize_pad(row_span,  IMG_SIZE, pad_val=0, is_mask=True)
          col_span,  _ = resize_pad(col_span,  IMG_SIZE, pad_val=0, is_mask=True)

          # =====================
          # 🔢 NORMALIZE
          # =====================
          img = img.astype(np.float32) / 255.0

          row_space = (row_space > 0).astype(np.float32)
          col_space = (col_space > 0).astype(np.float32)
          row_span  = (row_span  > 0).astype(np.float32)
          col_span  = (col_span  > 0).astype(np.float32)

          # =====================
          # 🔁 TO TENSOR
          # =====================
          img = img.transpose(2, 0, 1)   # (3,H,W)

          x = np.concatenate([
              img,
              row_space[None, :, :],
              col_space[None, :, :]
          ], axis=0)                     # (5,H,W)

          y = np.stack([
              row_span,
              col_span
          ], axis=0)                     # (2,H,W)

          return (
              torch.tensor(x, dtype=torch.float32),
              torch.tensor(y, dtype=torch.float32)
          )
        except Exception as e:
          return None



In [ ]:
def collate_skip_none(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return torch.utils.data.default_collate(batch)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        """
        patience: số epoch chờ nếu val_loss không cải thiện
        min_delta: mức cải thiện tối thiểu để coi là tốt hơn
        """
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return True   # có cải thiện
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
            return False

In [ ]:
import segmentation_models_pytorch as smp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 30
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=5,
    classes=2
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()
early_stopping = EarlyStopping(
    patience=10,      # tuỳ dataset, thường 8–15
    min_delta=1e-4
)

from segmentation_models_pytorch.losses import DiceLoss, FocalLoss

loss_col_fn = DiceLoss(mode="binary")

loss_row_fn = DiceLoss(mode="binary") + FocalLoss(
    mode="binary",
    alpha=0.75,
    gamma=2.0
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

/tmp/ipython-input-2793854345.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Using device:", DEVICE)

CUDA available: True
CUDA version: 12.6
GPU: Tesla T4
Using device: cuda


In [ ]:
from torch.utils.data import DataLoader

train_dataset = SpanTableDataset(
    img_dir="/content/images/train",
    row_space_dir="/content/row_space_masks/train",
    col_space_dir="/content/col_space_masks/train",
    row_span_dir="/content/row_span_masks/train",
    col_span_dir="/content/col_span_masks/train",
    augment=True
)


val_dataset = SpanTableDataset(
    img_dir="/content/images/val",
    row_space_dir="/content/row_space_masks/val",
    col_space_dir="/content/col_space_masks/val",
    row_span_dir="/content/row_span_masks/val",
    col_span_dir="/content/col_span_masks/val",
    augment=False
)

import numpy as np
from torch.utils.data import WeightedRandomSampler

row_positive_indices = []

for i in range(len(train_dataset)):
    _, mask = train_dataset[i]
    if mask[1].sum() > 0:  # channel row span
        row_positive_indices.append(i)

weights = []
for i in range(len(train_dataset)):
    if i in row_positive_indices:
        weights.append(5.0)   # oversample row span
    else:
        weights.append(1.0)

sampler = WeightedRandomSampler(
    weights,
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=False,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_skip_none
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_skip_none
)


In [ ]:
import torch, gc, os, signal

def release_gpu():
    print("Releasing GPU...")

    g = globals()

    if "model" in g:
        g["model"].to("cpu")
        del g["model"]

    if "optimizer" in g:
        del g["optimizer"]

    if "scaler" in g:
        del g["scaler"]

    torch.cuda.empty_cache()
    gc.collect()

    print("GPU released. Killing runtime...")
    os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
from tqdm import tqdm
import torch
import os

SAVE_DIR = "/content/drive/MyDrive/checkpoints_span_model"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_PATH = os.path.join(SAVE_DIR, "best.pth")
LAST_PATH = os.path.join(SAVE_DIR, "last.pth")

best_loss = float("inf")

print("Testing dataloader...")
imgs, masks = next(iter(train_loader))
print("OK:", imgs.shape, masks.shape)

for epoch in range(NUM_EPOCHS):

    # =========================
    # 🔥 TRAIN
    # =========================
    model.train()
    train_loss = 0.0
    train_total = 0
    train_skipped = 0
    train_used = 0

    train_pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS} [TRAIN]",
        leave=False
    )

    for batch in train_pbar:
        if batch is None:
            continue

        imgs, masks = batch
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.cuda.amp.autocast():
            preds = model(imgs)

            # tách channel
            pred_col = preds[:, 0:1, :, :]
            pred_row = preds[:, 1:2, :, :]

            gt_col = masks[:, 0:1, :, :]
            gt_row = masks[:, 1:2, :, :]

            loss_col = loss_col_fn(pred_col, gt_col)
            loss_row = loss_row_fn(pred_row, gt_row)

            # row span hiếm -> weight lớn
            LAMBDA_ROW = 10.0

            loss = loss_col + LAMBDA_ROW * loss_row
    
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_loader)
    print(
        f"[Epoch {epoch+1}] "
        f"TRAIN total={train_total}, "
        f"used={train_used}, "
        f"skipped={train_skipped}"
    )

    # =========================
    # 🧪 VALIDATION
    # =========================
    model.eval()
    val_loss = 0.0
    val_total = 0
    val_skipped = 0
    val_used = 0

    val_pbar = tqdm(
        val_loader,
        desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS} [VAL]",
        leave=False
    )

    with torch.no_grad():
        for batch in val_pbar:
            if batch is None:
                continue

            imgs, masks = batch
            imgs  = imgs.to(DEVICE)
            masks = masks.to(DEVICE)

            preds = model(imgs)
            loss  = loss_fn(preds, masks)

            val_loss += loss.item()
            val_pbar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss /= len(val_loader)

    print(
        f"[Epoch {epoch+1}] "
        f"VAL   total={val_total}, "
        f"used={val_used}, "
        f"skipped={val_skipped}"
    )
    # =========================
    # 📊 LOG
    # =========================
    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss = {train_loss:.4f} | "
        f"val_loss = {val_loss:.4f}"
    )

    # =========================
    # 💾 SAVE LAST
    # =========================
    torch.save({
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_loss": train_loss,
        "scaler": scaler.state_dict(),
        "val_loss": val_loss
    }, LAST_PATH)

    # =========================
    # ⭐ SAVE BEST (theo val_loss)
    # =========================
    improved = early_stopping(val_loss)

    if improved:
        torch.save({
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": val_loss
        }, BEST_PATH)
        print(f" ⭐ New BEST model (val_loss={val_loss:.4f})")

    if early_stopping.stop:
        print(
            f" ⛔ Early stopping triggered at epoch {epoch+1} "
            f"(best_val_loss={early_stopping.best_loss:.4f})"
        )
        break


release_gpu()
os.kill(os.getpid(), signal.SIGKILL)
print("🧊 GPU released. You can safely go to sleep.")

Testing dataloader...
OK: torch.Size([8, 5, 512, 512]) torch.Size([8, 2, 512, 512])


Epoch 01/30 [TRAIN]:   0%|          | 0/2573 [00:00<?, ?it/s]/tmp/ipython-input-113792297.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 01 | train_loss = 0.5858 | val_loss = 0.1398
 ⭐ New BEST model (val_loss=0.1398)


Epoch 02 | train_loss = 0.3641 | val_loss = 0.1221
 ⭐ New BEST model (val_loss=0.1221)


Epoch 03 | train_loss = 0.3329 | val_loss = 0.1206
 ⭐ New BEST model (val_loss=0.1206)


...

Epoch 17 | train_loss = 0.2146 | val_loss = 0.0703
 ⭐ New BEST model (val_loss=0.0703)


Epoch 18 | train_loss = 0.2097 | val_loss = 0.0732


Epoch 19 | train_loss = 0.2070 | val_loss = 0.0843


Epoch 20 | train_loss = 0.2016 | val_loss = 0.0689
 ⭐ New BEST model (val_loss=0.0689)


Epoch 21/30 [TRAIN]:   0%|          | 2/2573 [00:00<19:41,  2.18it/s, loss=0.0582]

Nếu đang train mà bị ngắt thì chạy các cell này, bỏ qua cell train trên

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/checkpoints"
LAST_PATH = f"{SAVE_DIR}/last.pth"
BEST_PATH = f"{SAVE_DIR}/best.pth"

In [ ]:
start_epoch = 0
best_loss = float("inf")

if os.path.exists(LAST_PATH):
    ckpt = torch.load(LAST_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"]
    best_loss = ckpt.get("val_loss", ckpt.get("loss", float("inf")))
    if "scaler" in ckpt:
      scaler.load_state_dict(ckpt["scaler"])
    print(f"✅ Resume from epoch {start_epoch}")
else:
    print("⚠️ Train from scratch")

In [ ]:
for epoch in range(start_epoch, NUM_EPOCHS):

    # =========================
    # 🔥 TRAIN
    # =========================
    model.train()
    train_loss = 0.0
    train_total = 0
    train_skipped = 0
    train_used = 0

    train_pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS} [TRAIN]",
        leave=False
    )

    for batch in train_pbar:
        if batch is None:
            continue

        imgs, masks = batch
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.cuda.amp.autocast():
            preds = model(imgs)

            # tách channel
            pred_col = preds[:, 0:1, :, :]
            pred_row = preds[:, 1:2, :, :]

            gt_col = masks[:, 0:1, :, :]
            gt_row = masks[:, 1:2, :, :]

            loss_col = loss_col_fn(pred_col, gt_col)
            loss_row = loss_row_fn(pred_row, gt_row)

            # row span hiếm -> weight lớn
            LAMBDA_ROW = 10.0

            loss = loss_col + LAMBDA_ROW * loss_row
    
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_loader)
    print(
        f"[Epoch {epoch+1}] "
        f"TRAIN total={train_total}, "
        f"used={train_used}, "
        f"skipped={train_skipped}"
    )

    # =========================
    # 🧪 VALIDATION
    # =========================
    model.eval()
    val_loss = 0.0
    val_total = 0
    val_skipped = 0
    val_used = 0

    val_pbar = tqdm(
        val_loader,
        desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS} [VAL]",
        leave=False
    )

    with torch.no_grad():
        for batch in val_pbar:
            if batch is None:
                continue

            imgs, masks = batch
            imgs  = imgs.to(DEVICE)
            masks = masks.to(DEVICE)

            preds = model(imgs)
            loss  = loss_fn(preds, masks)

            val_loss += loss.item()
            val_pbar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss /= len(val_loader)

    print(
        f"[Epoch {epoch+1}] "
        f"VAL   total={val_total}, "
        f"used={val_used}, "
        f"skipped={val_skipped}"
    )
    # =========================
    # 📊 LOG
    # =========================
    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss = {train_loss:.4f} | "
        f"val_loss = {val_loss:.4f}"
    )

    # =========================
    # 💾 SAVE LAST
    # =========================
    torch.save({
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_loss": train_loss,
        "scaler": scaler.state_dict(),
        "val_loss": val_loss
    }, LAST_PATH)

    # =========================
    # ⭐ SAVE BEST (theo val_loss)
    # =========================
    improved = early_stopping(val_loss)

    if improved:
        torch.save({
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": val_loss
        }, BEST_PATH)
        print(f" ⭐ New BEST model (val_loss={val_loss:.4f})")

    if early_stopping.stop:
        print(
            f" ⛔ Early stopping triggered at epoch {epoch+1} "
            f"(best_val_loss={early_stopping.best_loss:.4f})"
        )
        break

        
